# 📄 Day 02a: Paging — Page Tables, TLB & Multi-Level Paging

---

## 🎯 What You'll Master Today
- Paging concept — pages & frames
- Address translation (page number → frame number)
- Page table structure and TLB (Translation Lookaside Buffer)
- Multi-level page tables

---

## 📚 The Library Analogy

Paging is like a library's catalog system:
- **Page** = a chapter of your book (logical)
- **Frame** = a shelf slot in the library (physical)
- **Page table** = catalog card: "Chapter 3 is on Shelf 7"
- **TLB** = your personal sticky note of recently looked-up shelves

```
╔══════════════════════════════════════════════════════════╗
║              PAGING ADDRESS TRANSLATION                   ║
╠══════════════════════════════════════════════════════════╣
║                                                          ║
║  Logical Address:  | Page Number | Offset |              ║
║                        ↓                                 ║
║                   Page Table                              ║
║                   [P0 → F5]                               ║
║                   [P1 → F2]                               ║
║                   [P2 → F8]                               ║
║                        ↓                                 ║
║  Physical Address: | Frame Number | Offset |             ║
║                                                          ║
║  Page size = Frame size (always equal)                   ║
╚══════════════════════════════════════════════════════════╝
```

---

In [ ]:
# ============================================
# 1. Address Translation Simulator
# ============================================

class PagingSimulator:
    def __init__(self, page_size: int, page_table: dict[int, int]):
        self.page_size = page_size
        self.page_table = page_table  # {page_num: frame_num}
    
    def translate(self, logical_address: int) -> int:
        page_num = logical_address // self.page_size
        offset = logical_address % self.page_size
        
        print(f"  Logical Address: {logical_address}")
        print(f"    Page Number: {logical_address} / {self.page_size} = {page_num}")
        print(f"    Offset:      {logical_address} % {self.page_size} = {offset}")
        
        if page_num not in self.page_table:
            print(f"    ❌ PAGE FAULT! Page {page_num} not in memory")
            return -1
        
        frame_num = self.page_table[page_num]
        physical = frame_num * self.page_size + offset
        print(f"    Frame:       Page {page_num} → Frame {frame_num}")
        print(f"    Physical:    {frame_num} × {self.page_size} + {offset} = {physical}")
        return physical
    
    def show_table(self):
        print("  Page Table:")
        print("  Page# → Frame#")
        for p, f in sorted(self.page_table.items()):
            print(f"    [{p}]  →  [{f}]")


# Page size = 4KB, page table maps pages to frames
sim = PagingSimulator(
    page_size=4096,
    page_table={0: 5, 1: 2, 2: 8, 3: 1}
)
sim.show_table()
print()

# Translate some addresses
for addr in [0, 4096, 8192, 13000]:
    sim.translate(addr)
    print()

In [ ]:
# ============================================
# 2. TLB Simulation — Cache for Page Table
# ============================================

from collections import OrderedDict

class TLB:
    """Translation Lookaside Buffer with LRU eviction."""
    
    def __init__(self, capacity: int, page_table: dict[int, int]):
        self.capacity = capacity
        self.cache = OrderedDict()  # page_num → frame_num
        self.page_table = page_table
        self.hits = 0
        self.misses = 0
    
    def lookup(self, page_num: int) -> int:
        if page_num in self.cache:
            self.hits += 1
            self.cache.move_to_end(page_num)  # LRU update
            print(f"    TLB HIT ✅  Page {page_num} → Frame {self.cache[page_num]}")
            return self.cache[page_num]
        
        # TLB miss — go to page table
        self.misses += 1
        frame = self.page_table.get(page_num, -1)
        print(f"    TLB MISS ❌ Page {page_num} → Frame {frame} (loaded from page table)")
        
        if frame != -1:
            if len(self.cache) >= self.capacity:
                evicted = self.cache.popitem(last=False)  # evict LRU
                print(f"    Evicted Page {evicted[0]} from TLB")
            self.cache[page_num] = frame
        
        return frame
    
    def stats(self):
        total = self.hits + self.misses
        ratio = (self.hits / total * 100) if total else 0
        print(f"\n  📊 TLB Stats: {self.hits} hits, {self.misses} misses, {ratio:.1f}% hit rate")


# TLB with capacity 3, full page table behind it
page_table = {0: 5, 1: 2, 2: 8, 3: 1, 4: 7, 5: 3}
tlb = TLB(capacity=3, page_table=page_table)

# Access pattern: locality matters!
access_sequence = [0, 1, 2, 0, 1, 3, 0, 3, 2, 4]
print("  Page Access Sequence:", access_sequence)
print()

for page in access_sequence:
    tlb.lookup(page)

tlb.stats()

In [ ]:
# ============================================
# 3. Multi-Level Paging — Address Breakdown
# ============================================

def multilevel_paging_demo():
    """
    Two-level paging address translation.
    32-bit address, 4KB pages:
      [10-bit L1 index | 10-bit L2 index | 12-bit offset]
    """
    print("  Two-Level Paging (32-bit, 4KB pages)")
    print("  " + "═" * 45)
    print("  Address format:")
    print("  |  10 bits  |  10 bits  |  12 bits  |")
    print("  | L1 Index  | L2 Index  |  Offset   |")
    print()
    
    # Simulate two-level page table
    # L1 table: index → L2 table reference
    # L2 table: index → frame number
    l2_tables = {
        0: {0: 10, 1: 23, 5: 42},  # L2 table pointed by L1[0]
        3: {2: 15, 7: 31},         # L2 table pointed by L1[3]
    }
    
    # Translate address
    logical = 0x00C01A00  # binary: 0000 0000 1100 | 0000 0001 10 | 10 0000 0000
    
    offset = logical & 0xFFF           # lower 12 bits
    l2_index = (logical >> 12) & 0x3FF # next 10 bits
    l1_index = (logical >> 22) & 0x3FF # upper 10 bits
    
    print(f"  Logical Address: 0x{logical:08X} ({logical})")
    print(f"    L1 Index: {l1_index}")
    print(f"    L2 Index: {l2_index}")
    print(f"    Offset:   {offset}")
    print()
    
    # Why multi-level?
    print("  💡 Why multi-level paging?")
    print("  ─" * 25)
    single_table = 2**20 * 4  # 1M entries × 4 bytes each
    print(f"    Single-level: {single_table // 1024}KB per process (wasteful!)")
    print(f"    Two-level: only allocate L2 tables for used address ranges")
    print(f"    L1 table: {2**10 * 4 // 1024}KB + only needed L2 tables")


multilevel_paging_demo()

## 📝 Interview Questions

| # | Question | Key Point |
|---|----------|------------|
| 1 | Why does paging eliminate external fragmentation? | Every page fits any frame — no need for contiguous blocks |
| 2 | What causes internal fragmentation in paging? | Last page may not be fully used. Max waste = page_size - 1 |
| 3 | What is a TLB and why is it needed? | Cache for page table lookups. Without it, every memory access needs 2 accesses |
| 4 | TLB miss penalty? | Must access page table in memory. Effective access = TLB_time + (miss_rate × memory_access) |
| 5 | Why multi-level page tables? | Single-level for 32-bit = 4MB per process. Multi-level only allocates what's used |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • Page = logical block, Frame = physical block (=size) │
## │  • Physical = frame_num × page_size + offset            │
## │  • TLB = fast cache, 95%+ hit rate with locality        │
## │  • Multi-level saves space: only allocate used L2 tables│
## │  • Paging eliminates external frag, tiny internal frag  │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **Virtual Memory** — demand paging, page faults & thrashing